# Global Sensitivity Analysis (Sobol & Borgonovo)

In [ ]:
from SALib.sample import saltelli
from SALib.analyze import sobol
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(" STEP 3: GLOBAL SENSITIVITY ANALYSIS (SOBOL) ")

# Define the SALib problem dictionary based on the bounds of the 15 microparameters
problem = {
    'num_vars': len(X_train.columns),
    'names': list(X_train.columns),
    'bounds': [[X_train[col].min(), X_train[col].max()] for col in X_train.columns]
}

# Generate samples for Sobol analysis
param_values = saltelli.sample(problem, 1024)
param_scaled = scaler_X.transform(param_values)

# Evaluate the Kriging model for the generated samples
Y_pred = multi_gpr.predict(param_scaled)
Y_ucs = scaler_y.inverse_transform(Y_pred)[:, 0]

# Perform Sobol analysis for Peak UCS
Si_ucs = sobol.analyze(problem, Y_ucs, print_to_console=False)

# Extract and visualize First-Order (S1) and Total-Order (ST) Sensitivity Indices
sobol_df = pd.DataFrame({
    'Parameter': problem['names'],
    'S1': Si_ucs['S1'],
    'ST': Si_ucs['ST']
}).sort_values(by='ST', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(sobol_df['Parameter'][::-1], sobol_df['ST'][::-1], color='steelblue', label='Total-Order (ST)')
plt.barh(sobol_df['Parameter'][::-1], sobol_df['S1'][::-1], color='orange', label='First-Order (S1)')
plt.xlabel('Sensitivity Index')
plt.title('Sobol Sensitivity Analysis for Peak UCS')
plt.legend()
plt.tight_layout()
plt.show()

print("\nMost Sensitive Parameters (ST > 0.1):")
print(sobol_df[sobol_df['ST'] > 0.1])
